In [1]:
import collections

class Vocabulary:
    def __init__(self, pad_token="<pad>", unk_token="<unk>", min_freq=1):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.min_freq = min_freq

        self.w2i = {self.pad_token: 0, self.unk_token: 1}
        self.i2w = {0: self.pad_token, 1: self.unk_token}

    def build_vocab(self, texts):
        """Сканирует все тексты, считает частоту слов и добавляет из в словарь"""
        counter = collections.Counter()
        for text in texts:
            # Простейшая токенизация по пробелам
            counter.update(text.lower().split())

        for word, freq in counter.items():
            if freq >= self.min_freq and word not in self.w2i:
                idx = len(self.w2i)
                self.w2i[word] = idx
                self.i2w[idx] = word

    def encode(self, text):
        tokens = text.lower().split()
        return [self.w2i.get(token, self.w2i[self.unk_token]) for token in tokens]

    def decode(self, indices):
        return [self.i2w.get(idx, self.unk_token) for idx in indices]

    def __len__(self):
        return len(self.w2i)

sample_texts = [
    "заблокировали дебетовую карту срочно",
    "украли деньги со счета мошенники",
    "как узнать баланс карты"
]

vocab = Vocabulary(min_freq=1)
vocab.build_vocab(sample_texts)

print(f"Размер построенного словаря: {len(vocab)} токенов")
print("Внутренний словарь (word -> index):", vocab.w2i)

new_text = "украли карту в банкомате"
encoded = vocab.encode(new_text)
decoded = vocab.decode(encoded)

print(f"\nИсходный текст: '{new_text}'")
print(f"Закодированные ID: {encoded}")
print(f"Декодированные обратно токены: {decoded}")

Размер построенного словаря: 15 токенов
Внутренний словарь (word -> index): {'<pad>': 0, '<unk>': 1, 'заблокировали': 2, 'дебетовую': 3, 'карту': 4, 'срочно': 5, 'украли': 6, 'деньги': 7, 'со': 8, 'счета': 9, 'мошенники': 10, 'как': 11, 'узнать': 12, 'баланс': 13, 'карты': 14}

Исходный текст: 'украли карту в банкомате'
Закодированные ID: [6, 4, 1, 1]
Декодированные обратно токены: ['украли', 'карту', '<unk>', '<unk>']


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

class BankingDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.labels = list(labels)
        self.encoded_texts = [vocab.encode(t) for t in texts]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded_texts[idx], self.labels[idx]

def collate_fn(batch):
    """
    Собирает список сэмплов из __getitem__ в батч
    Для EmbeddingBag тексты склеиваются в один длинный тензор,
    а offsets указывают градиенты/границы
    """

    text_list, label_list, offsets = [], [], [0]

    for _text, _label in batch:
        label_list.append(_label)
        processed_text = torch.tensor(_text, dtype=torch.long)
        text_list.append(processed_text)
        # запоминаем длину текущего текста для смещений (offsets)
        offsets.append(processed_text.size(0))

    label_list = torch.tensor(label_list, dtype=torch.long)
    # превращаем длины в кумулятивную сумму (напрaвления смещений)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    # склеиваем все тексты батча в один плоский 1D-тензор
    text_list = torch.cat(text_list)

    return text_list, offsets, label_list

sample_texts = ["заблокировали карту", "украли деньги", "почему не работает"]
sample_labels = [0, 1, 2]

dataset = BankingDataset(sample_texts, sample_labels, vocab)
print(f"Длина датасета: {len(dataset)}")
print("Пример работы __getitem__(0):", dataset[0])

dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

for text_flat, offsets, targets in dataloader:
    print("\nРезультат работы DataLoader (один батч):")
    print("Плоский тензор токенов (text_flat):", text_flat)
    print("Смещения текстов (offsets):", offsets)
    print("Метки классов (targets):", targets)
    break

Длина датасета: 3
Пример работы __getitem__(0): ([2, 4], 0)

Результат работы DataLoader (один батч):
Плоский тензор токенов (text_flat): tensor([2, 4, 6, 7])
Смещения текстов (offsets): tensor([0, 2])
Метки классов (targets): tensor([0, 1])


In [3]:
import torch
import torch.nn as nn

class DeepTextMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden1, hidden2, num_classes, dropout=0.2):
        super().__init__()

        # слой эмбеддингов с усреднением по длине текста(входной)
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean")

        # скрытый слой 1
        self.fc1 = nn.Linear(embed_dim, hidden1)
        self.act1 = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)

        # скрытый слой 2
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.act2 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)

        # выходной слой(логиты каждого класса)
        self.fc3 = nn.Linear(hidden2, num_classes)

        # инилизация вессов (Kaiming Normal для стабильного старта)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, text, offsets):
        """Прямой ход"""
        # превращаем текст в эмбеддинг [batch_size, embed_dim]
        x = self.embedding(text, offsets)

        # прогоняем через скрытые слои
        x = self.fc1(x)
        x = self.act1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.act2(x)
        x = self.dropout2(x)

        logits = self.fc3(x)
        return logits

model = DeepTextMLP(
    vocab_size = len(vocab),
    embed_dim=32,
    hidden1 = 16,
    hidden2 = 8,
    num_classes = 3
)

# прогоняем тестовый батч
for text_flat, offsets, targets in dataloader:
    logits = model(text_flat, offsets)
    print("Размер выходных логитов модели:", logits.shape) # ожидаем [2, 3]
    print("Сырые логиты классов:",logits)



Размер выходных логитов модели: torch.Size([2, 3])
Сырые логиты классов: tensor([[ 0.6932, -0.3221, -0.7183],
        [ 0.0493, -0.1192,  0.0131]], grad_fn=<AddmmBackward0>)
Размер выходных логитов модели: torch.Size([1, 3])
Сырые логиты классов: tensor([[-0.4177,  0.8855,  0.3679]], grad_fn=<AddmmBackward0>)


In [5]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# 1. Инилизация компонентов оптимизации

# переносим модель на GPU при наличии
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# функция потерь
criterion = nn.CrossEntropyLoss()

# оптимизатор
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# планировщик скорости обучения на 5 эпох
epochs = 5
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

# 2. Функция одной эпохи обучения (Train Loop)
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # активирует dropout
    total_loss = 0.0

    for text_flat, offsets, targets in dataloader:
        # переносим тензоры на нужное устройство
        text_flat = text_flat.to(device)
        offsets = offsets.to(device)
        targets = targets.to(device)

        # 1) обнуляем градиенты с прошлого шага
        optimizer.zero_grad()

        # 2) forward pass получаем предсказания
        logits = model(text_flat, offsets)

        # 3) вычисляем ошибку
        loss = criterion(logits, targets)

        # 4) backward pass автодифференцирование (считаем градиенты dL/dw)
        loss.backward()

        # 5) клиппинг градиентов (защита от вылетов)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 6) корректируем веса модели
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 3. Тестируем модель
for epoch in range(1, epochs+1):
    # обучаем модель одну эпоху
    loss = train_epoch(model, dataloader, criterion, optimizer, device)

    # обновляем LR на следующий шаг
    scheduler.step()

    # извлекаем текущий LR для записи
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Эпоха [{epoch}/{epochs}] | Loss: {loss:.4f} | LR: {current_lr:.6f}")

Эпоха [1/5] | Loss: 0.9749 | LR: 0.000905
Эпоха [2/5] | Loss: 1.5111 | LR: 0.000655
Эпоха [3/5] | Loss: 0.9027 | LR: 0.000345
Эпоха [4/5] | Loss: 1.2826 | LR: 0.000095
Эпоха [5/5] | Loss: 0.5778 | LR: 0.000000
